# A multi-index RAG pipeline

Semantic search finds meaning, BM25 finds exact terms. A **`Retriever`** runs both and merges the results.

The scores can't be compared directly: a cosine distance of 0.6 and a BM25 distance of 0.3 are on
different scales. **Reciprocal Rank Fusion (RRF)** uses only the ranks:

```
RRF_score(doc) = sum over indexes of 1 / (k + rank of doc in that index)
```

A document near the top of several lists gets a high total. `k` (60 by default) flattens the
difference between ranks; the lesson's example uses 1 to keep the numbers simple.

The class is in [`retriever.py`](./retriever.py): it passes every document to all indexes, asks each
for `k * 5` results and merges them with RRF.

## RRF by hand

The lesson's example, with `k = 1`.

In [1]:
vector_ranks = {"Section 2": 1, "Section 7": 2, "Section 6": 3}
bm25_ranks = {"Section 6": 1, "Section 2": 2, "Section 7": 3}
k = 1

for section, vector_rank in vector_ranks.items():
    bm25_rank = bm25_ranks[section]
    score = 1 / (k + vector_rank) + 1 / (k + bm25_rank)
    print(f"{section}: 1/({k}+{vector_rank}) + 1/({k}+{bm25_rank}) = {score:.3f}")

Section 2: 1/(1+1) + 1/(1+2) = 0.833
Section 7: 1/(1+2) + 1/(1+3) = 0.583
Section 6: 1/(1+3) + 1/(1+1) = 0.750


Section 2 wins: it is near the top of both lists, while Section 6 is first in only one.

## Setup

In [2]:
from pathlib import Path

from bm25_index import BM25Index
from chunking import chunk_by_section
from embeddings import generate_embedding
from retriever import Retriever
from vector_index import VectorIndex

chunks = chunk_by_section(Path("report.md").read_text())

vector_index = VectorIndex(embedding_fn=generate_embedding)
bm25_index = BM25Index()
retriever = Retriever(bm25_index, vector_index)

# one call: the retriever passes the documents on to both indexes, the vector index embeds them in one request
retriever.add_documents([{"content": chunk} for chunk in chunks])
print(vector_index, bm25_index, sep="\n")

VectorIndex(count=15, dim=1024, metric='cosine', has_embedding_fn='Yes')
BM25Index(count=15, k1=1.5, b=0.75)


## Hybrid search

In [3]:
results = retriever.search("What happened with INC-2023-Q4-011?", k=3)

for doc, score in results:
    print(f"{score:.4f}  {doc['content'][:120]}...")

0.0325  ## Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated...
0.0325  ## Section 10: Cybersecurity Analysis - Incident Response Report: INC-2023-Q4-011

The Cybersecurity Operations Center s...
0.0304  ## Methodology

The insights compiled within this Annual Interdisciplinary Research Review represent a synthesis of find...


The two incident sections tie at 0.0325: one is 1st in BM25 and 2nd in vector search, the other the
other way round, and `1/61 + 1/62` is the same both ways. With `k=60` scores are small and close;
only the order matters.

## Does merging help?

The rank of the right section in each index and after RRF, for the four questions from the BM25 lesson.

In [4]:
# each question with the section that answers it
questions = {
    "What happened with INC-2023-Q4-011?": "## Section 10:",
    "TIMEOUT_QUERY_DB_0xDEADBEEF": "## Section 2:",
    "What did the software engineering dept do last year?": "## Section 2:",
    "Which team stopped hackers from breaking in?": "## Section 10:",
}


def rank_of(results, prefix):
    ranks = [rank for rank, (doc, _) in enumerate(results, start=1) if doc["content"].startswith(prefix)]
    return str(ranks[0]) if ranks else "-"


n = len(chunks)
print(f"{'question':54} {'vector':>6} {'bm25':>5} {'rrf':>4} {'rrf k=1':>8}")
for question, answer in questions.items():
    print(
        f"{question:54} {rank_of(vector_index.search(question, k=n), answer):>6}"
        f" {rank_of(bm25_index.search(question, k=n), answer):>5}"
        f" {rank_of(retriever.search(question, k=n), answer):>4}"
        f" {rank_of(retriever.search(question, k=n, k_rrf=1), answer):>8}"
    )

question                                               vector  bm25  rrf  rrf k=1
What happened with INC-2023-Q4-011?                         1     2    2        2
TIMEOUT_QUERY_DB_0xDEADBEEF                                 1     1    1        1
What did the software engineering dept do last year?        2     3    2        2
Which team stopped hackers from breaking in?                1     -   13        3


- **When both indexes agree**, RRF keeps the right answer on top.
- **When one index is right and the other has nothing**, RRF can bury the answer. The hacker question:
  vector search puts the incident report 1st, BM25 doesn't return it at all, and after the merge it
  falls far down. With `k=60`, one first place gives `1/61 ≈ 0.016`, while two mediocre places
  give more (`2/70 ≈ 0.029`). A smaller `k` rewards the top places more and helps a bit.
- **Hybrid search is not automatically better.** It pays off when the questions contain exact terms
  and the embedding model misses them. Check it on your own questions, the way module 2 evaluated prompts.

Next lesson: extended thinking.